In [ ]:
 from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset_path = "/content/drive/MyDrive/pan11"

In [ ]:
import os
print(os.listdir(dataset_path)[:10])

['external-detection-corpus']


In [ ]:
import glob

suspicious_files = glob.glob(dataset_path + "/**/suspicious-document*.txt", recursive=True)
source_files = glob.glob(dataset_path + "/**/source-document*.txt", recursive=True)

print("Suspicious docs:", len(suspicious_files))
print("Source docs:", len(source_files))

Suspicious docs: 4000
Source docs: 500


In [ ]:
def read_file(path):
    try:
        with open(path, 'r', encoding='latin-1') as f:
            return f.read()
    except:
        return ""

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_similarity(doc1, doc2):
    vectorizer = TfidfVectorizer(max_features=5000)
    vectors = vectorizer.fit_transform([doc1, doc2])
    return cosine_similarity(vectors[0], vectors[1])[0][0]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def bert_similarity(doc1, doc2):
    embeddings = model.encode([doc1, doc2])
    return cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from nltk.tokenize import sent_tokenize

def sentence_level_similarity(doc1, doc2):
    sentences1 = sent_tokenize(doc1)
    sentences2 = sent_tokenize(doc2)

    results = []

    for s1 in sentences1:
        max_score = 0
        best_match = ""

        for s2 in sentences2:
            score = bert_similarity(s1, s2)
            if score > max_score:
                max_score = score
                best_match = s2

        results.append((s1, best_match, max_score))

    return results

In [ ]:
def hybrid_similarity(doc1, doc2, alpha=0.3):
    tfidf_score = tfidf_similarity(doc1, doc2)
    bert_score = bert_similarity(doc1, doc2)

    return alpha * tfidf_score + (1 - alpha) * bert_score

In [ ]:
doc1 = preprocess(read_file(suspicious_files[0]))
doc2 = preprocess(read_file(source_files[0]))

print("TF-IDF:", tfidf_similarity(doc1, doc2))
print("BERT:", bert_similarity(doc1, doc2))
print("Hybrid:", hybrid_similarity(doc1, doc2))

TF-IDF: 0.08905700821249955
BERT: 0.26414043
Hybrid: 0.21161540442278765


In [ ]:
import nltk
nltk.download('punkt_tab')

results = sentence_level_similarity(doc1, doc2)

threshold = 0.2

for s1, s2, score in results:
    if score > threshold:
        print("\nPlagiarism Detected")
        print("Sentence:", s1)
        print("Match:", s2)
        print("Score:", score)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
for i in range(5):
    doc1 = preprocess(read_file(suspicious_files[i]))
    doc2 = preprocess(read_file(source_files[i]))

    score = hybrid_similarity(doc1, doc2)
    print(f"Pair {i} → Score: {score}")

Pair 0 → Score: 0.21161540442278765
Pair 1 → Score: 0.2550919900774742
Pair 2 → Score: -0.012298809047473417
Pair 3 → Score: 0.24008045293842084
Pair 4 → Score: 0.28201909041967366


<h1>UPLOAD PDF AND CHECK</h1>

In [ ]:
import requests
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from PyPDF2 import PdfReader
from google.colab import files

nltk.download('punkt')

API_KEY = "043a97df7daacbacd11f3614355bacaec0d26707ea91cf0c327d39013058a769"

uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

print("Uploaded:", pdf_file)

def extract_pdf_text(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

input_text = extract_pdf_text(pdf_file)

print("\nSample Text:\n", input_text[:300])

model = SentenceTransformer('all-MiniLM-L6-v2')

def similarity(a, b):
    emb = model.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

def google_search(query):
    try:
        url = "https://serpapi.com/search"

        params = {
            "engine": "google",
            "q": query,
            "api_key": API_KEY,
            "num": 3
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code != 200:
            return []

        data = response.json()

        results = []

        if "organic_results" in data:
            for r in data["organic_results"]:
                if "snippet" in r:
                    results.append(r["snippet"])

        return results

    except:
        return []

input_sentences = sent_tokenize(input_text)

results = []

print("\nChecking plagiarism...\n")

for sent in input_sentences[:10]:

    query = sent[:200]

    web_results = google_search(query)

    if len(web_results) == 0:
        continue

    max_score = 0
    best_match = ""

    for web_text in web_results:
        score = similarity(sent, web_text)

        if score > max_score:
            max_score = score
            best_match = web_text

    results.append((sent, best_match, max_score))

if len(results) == 0:
    print("No results found from Google. Try different content.")
else:
    threshold = 0.5

    plag = [r for r in results if r[2] > threshold]

    plag_percent = (len(plag) / len(results)) * 100

    print(f"\nFINAL PLAGIARISM: {plag_percent:.2f}%")

    if len(plag) > 0:
        print("\nDetected Plagiarized Sentences:\n")

        for sent, match, score in plag:
            print("Input:", sent)
            print("Found:", match)
            print("Score:", score)
            print("="*100)

    else:
        print("\nNo strong plagiarism. Showing top matches:\n")

        results = sorted(results, key=lambda x: x[2], reverse=True)

        for sent, match, score in results[:5]:
            print("Input:", sent)
            print("Found:", match)
            print("Score:", score)
            print("="*100)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Saving Untitled document (5).pdf to Untitled document (5).pdf
Uploaded: Untitled document (5).pdf

📄 Sample Text:
 The
 
rise
 
of
 
artificial
 
intelligence
 
has
 
fundamentally
 
altered
 
the
 
landscape
 
of
 
modern
 
education,
 
providing
 
new
 
tools
 
for
 
personalized
 
learning
 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔍 Checking plagiarism...


🔥 FINAL PLAGIARISM: 100.00%

⚠️ Detected Plagiarized Sentences:

👉 Input: The
 
rise
 
of
 
artificial
 
intelligence
 
has
 
fundamentally
 
altered
 
the
 
landscape
 
of
 
modern
 
education,
 
providing
 
new
 
tools
 
for
 
personalized
 
learning
🌐 Found: AI's potential impact on education is multifaceted, encompassing personalized learning, intelligent tutoring systems, and data-driven insights.
📈 Score: 0.7725059
